# Biblical/Christian Glossary — PoC v3

**Change from v2:** the English source is now [CCEL](https://www.ccel.org)
(Christian Classics Ethereal Library), not biblestudytools.com.

Why the switch: biblestudytools.com's index page only exposed 379 links —
about 9.5% of the real dictionary (confirmed below: Easton's Bible
Dictionary, 1897, has **3,963 entries**, matching independent web-search
confirmation of "nearly 4,000"). CCEL hosts the complete dictionary as:
- one master index listing all headwords with stable anchor IDs,
- the actual definitions split across only **40 chunk pages** (~100 entries
  each) — so fetching the *entire* dictionary is ~41 HTTP requests, not
  thousands,
- directly `curl`-able with no anti-bot blocking (unlike sacred-texts.com's
  403, or having to hit biblestudytools.com once per word),
- clean, uniform markup — no nav/share-button text to strip out.

Still no Malayalam anywhere in this pipeline except the verse-lookup
assist step (Step 4, kept from v2), which surfaces real scripture text for
a human to read — it does not generate or assert a translation.


In [1]:
import re
import html as htmlmod
import time
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (research; linguaalayam biblical-glossary PoC)"}
CCEL_BASE = "https://www.ccel.org/e/easton/ebd/"

def fetch(url: str) -> str:
    resp = requests.get(url, headers=HEADERS, timeout=20)
    resp.raise_for_status()
    return resp.text


## Step 1 — fetch the complete master index

One request. Every headword in the dictionary, with the chunk file and
anchor ID where its definition actually lives.


In [2]:
master_html = fetch(CCEL_BASE + "ebd3.html")

# <DT><A HREF="ebd/T0000000.html#T0000002">Aaron</A>
index_entries = re.findall(r'<DT><A HREF="(ebd/T\d+)\.html#(T\d+)">([^<]+)</A>', master_html)

chunk_files = sorted({chunk for chunk, anchor, headword in index_entries})
print(f"Discovered {len(index_entries)} headwords across {len(chunk_files)} chunk files.")
print("First 5:", index_entries[:5])
print("Last 5: ", index_entries[-5:])


Discovered 3963 headwords across 40 chunk files.
First 5: [('ebd/T0000000', 'T0000002', 'Aaron'), ('ebd/T0000000', 'T0000003', 'Aaronites'), ('ebd/T0000000', 'T0000004', 'Abaddon'), ('ebd/T0000000', 'T0000005', 'Abagtha'), ('ebd/T0000000', 'T0000006', 'Abana')]
Last 5:  [('ebd/T0003900', 'T0003960', 'Zuph, Land of'), ('ebd/T0003900', 'T0003961', 'Zur'), ('ebd/T0003900', 'T0003962', 'Zuriel'), ('ebd/T0003900', 'T0003963', 'Zurishaddai'), ('ebd/T0003900', 'T0003964', 'Zuzims')]


**Completeness check:** 3,963 discovered vs. the ~3,964/"nearly 4,000"
figure independently reported for this dictionary — accounted for
(one likely lost to the single-letter "A" section-heading entry, which
this regex intentionally skips since it has no anchor). This is now a
genuinely complete list, not a partial sample.


## Step 2 — fetch all 40 chunk files and extract every entry, verbatim

Only 40 HTTP requests cover the entire dictionary's actual text.


In [3]:
def split_chunk_into_segments(chunk_html: str) -> dict[str, str]:
    """Split a chunk page into {anchor_id: raw_html_until_next_anchor}."""
    parts = re.split(r'<A NAME="(T\d+)">', chunk_html)
    segments = {}
    for i in range(1, len(parts), 2):
        anchor_id = parts[i]
        content = parts[i + 1] if i + 1 < len(parts) else ""
        segments[anchor_id] = content
    return segments


def clean_entry_text(raw_html: str, headword: str) -> str:
    text = re.sub(
        r'^\s*<B>\s*' + re.escape(headword) + r'\s*-?\s*</B>\s*',
        "", raw_html, count=1, flags=re.IGNORECASE,
    )
    text = re.sub(r"<[^>]+>", " ", text)
    text = htmlmod.unescape(text)
    return re.sub(r"\s+", " ", text).strip()


def extract_refs(definition: str) -> list[str]:
    refs = []
    for group in re.findall(r"\(([^()]*\d+:\d+[^()]*)\)", definition):
        refs.extend(re.findall(
            r"(?:[1-3]\s)?[A-Z][a-zA-Z.]*\.?\s*\d+:\d+(?:[-,]\s?\d+)*", group
        ))
    return refs


index_by_chunk: dict[str, list[tuple[str, str]]] = {}
for chunk, anchor, headword in index_entries:
    index_by_chunk.setdefault(chunk, []).append((anchor, headword))

records = []
for i, chunk in enumerate(chunk_files):
    chunk_html = fetch(f"https://www.ccel.org/e/easton/ebd/{chunk}.html")
    segments = split_chunk_into_segments(chunk_html)
    for anchor, headword in index_by_chunk[chunk]:
        raw = segments.get(anchor)
        if raw is None:
            continue
        definition = clean_entry_text(raw, headword)
        if not definition:
            continue
        records.append({
            "headword": headword,
            "definition": definition,
            "refs": extract_refs(definition),
            "ml_variants": [],   # deliberately empty -- requires a human
            "entry_kind": None,  # not reliably inferable from the source markup
            "source_url": f"https://www.ccel.org/e/easton/ebd/{chunk}.html#{anchor}",
        })
    if (i + 1) % 10 == 0 or i == len(chunk_files) - 1:
        print(f"  fetched {i + 1}/{len(chunk_files)} chunk files, {len(records)} entries so far")
    time.sleep(0.3)  # be polite to the source server

print(f"\nParsed {len(records)} entries total out of {len(index_entries)} indexed headwords.")


  fetched 10/40 chunk files, 998 entries so far


  fetched 20/40 chunk files, 1998 entries so far


  fetched 30/40 chunk files, 2998 entries so far


  fetched 40/40 chunk files, 3963 entries so far



Parsed 3963 entries total out of 3963 indexed headwords.


## Step 3 — inspect a few, verbatim, unlabelled, at real scale


In [4]:
import random
random.seed(0)
for rec in random.sample(records, 5):
    print(f"headword:    {rec['headword']}")
    print(f"refs:        {rec['refs']}")
    print(f"ml_variants: {rec['ml_variants']}  <-- empty, needs a human")
    print(f"definition:  {rec['definition'][:200]}{'...' if len(rec['definition']) > 200 else ''}")
    print(f"source:      {rec['source_url']}")
    print("-" * 80)

with_refs = sum(1 for r in records if r["refs"])
print(f"\n{len(records)} total entries; {with_refs} ({with_refs/len(records):.0%}) have at least one extracted scripture ref.")
print("(Refs extraction is regex-based and best-effort -- it misses implicit")
print(" same-book citations like \"(2:1,4; 7:7)\" that don't repeat the book name.")
print(" That's a known gap, not a bug to silently ignore.)")


headword:    Sling
refs:        ['1 Sam. 17:40, 49', 'Judg. 20:16', '1 Chr. 12:2', '2 Kings 3:25']
ml_variants: []  <-- empty, needs a human
definition:  With a sling and a stone David smote the Philistine giant (1 Sam. 17:40, 49). There were 700 Benjamites who were so skilled in its use that with the left hand they "could sling stones at a hair breadt...
source:      https://www.ccel.org/e/easton/ebd/ebd/T0003400.html#T0003460
--------------------------------------------------------------------------------
headword:    Hadoram
refs:        ['1 Chr. 18:10', '2 Sam. 8:10', 'Gen. 10:27', '1 Chr. 1:21', '2 Chr. 10:18', '2 Sam. 20:24', '1 Kings 4:6']
ml_variants: []  <-- empty, needs a human
definition:  is exalted. (1.) The son of Tou, king of Hamath, sent by his father to congratulate David on his victory over Hadarezer, king of Syria (1 Chr. 18:10; called Joram 2 Sam. 8:10). (2.) The fifth son of J...
source:      https://www.ccel.org/e/easton/ebd/ebd/T0001500.html#T0001579
------------

## The `ml_variants` schema (illustrative only — no real headword filled in)

Unchanged from v2 — still a list, not a single string, since one English
headword can have multiple valid Malayalam forms depending on
Latin/Catholic, Syriac Orthodox, or Protestant-Bible-translation tradition:

```python
{
    "headword": "<some English name>",
    "ml_variants": [
        {"spelling": "<option A>", "tradition": "Syriac Orthodox usage", "notes": "..."},
        {"spelling": "<option B>", "tradition": "Malayalam Protestant Bible (e.g. IRV/OV)", "notes": "..."},
        {"spelling": "<option C>", "tradition": "Catholic missal / calendar", "notes": "..."},
    ],
}
```


## Step 4 — Malayalam verse-lookup assist (human still decides)

This does **not** produce a Malayalam translation automatically. For each
English entry, it uses the scripture refs already extracted to pull the
*actual* Malayalam Bible verse(s) where that name/concept appears, from an
open, CC BY-SA-licensed source:
[FreeBiblesIndia/Malayalam_Bible](https://github.com/FreeBiblesIndia/Malayalam_Bible)
(USFM format).

The point is to save a human labeller the trouble of going and finding the
right verse themselves — they still have to *read* the Malayalam text and
pick out/confirm the correct word or phrase. Nothing here asserts a mapping;
it only surfaces the candidate context.


In [5]:
USFM_API = "https://api.github.com/repos/FreeBiblesIndia/Malayalam_Bible/contents/usfm"
RAW_BASE = "https://raw.githubusercontent.com/FreeBiblesIndia/Malayalam_Bible/master/usfm/"

# Build {USFM book code -> filename} once, from the live repo listing.
resp = requests.get(USFM_API, headers=HEADERS, timeout=15)
resp.raise_for_status()
usfm_files = {}
for item in resp.json():
    m = re.match(r"\d+_([0-9A-Z]+)MAL\.usfm", item["name"])
    if m:
        usfm_files[m.group(1)] = item["name"]

print(f"Found {len(usfm_files)} Malayalam USFM book files in the live repo.")

# English ref book-name -> USFM code. Standard 66-book Protestant canon.
BOOK_ALIASES = {
    "genesis": "GEN", "exodus": "EXO", "leviticus": "LEV", "numbers": "NUM",
    "deuteronomy": "DEU", "joshua": "JOS", "judges": "JDG", "ruth": "RUT",
    "1samuel": "1SA", "2samuel": "2SA", "1kings": "1KI", "2kings": "2KI",
    "1chronicles": "1CH", "2chronicles": "2CH", "ezra": "EZR", "nehemiah": "NEH",
    "esther": "EST", "job": "JOB", "psalms": "PSA", "psalm": "PSA",
    "proverbs": "PRO", "ecclesiastes": "ECC", "isaiah": "ISA", "jeremiah": "JER",
    "lamentations": "LAM", "ezekiel": "EZK", "daniel": "DAN", "hosea": "HOS",
    "joel": "JOL", "amos": "AMO", "obadiah": "OBA", "jonah": "JON",
    "micah": "MIC", "nahum": "NAM", "habakkuk": "HAB", "zephaniah": "ZEP",
    "haggai": "HAG", "zechariah": "ZEC", "malachi": "MAL",
    "matthew": "MAT", "matt": "MAT", "mark": "MRK", "luke": "LUK", "john": "JHN",
    "acts": "ACT", "romans": "ROM", "1corinthians": "1CO", "2corinthians": "2CO",
    "galatians": "GAL", "ephesians": "EPH", "philippians": "PHP",
    "colossians": "COL", "1thessalonians": "1TH", "2thessalonians": "2TH",
    "1timothy": "1TI", "2timothy": "2TI", "titus": "TIT", "philemon": "PHM",
    "hebrews": "HEB", "james": "JAS", "1peter": "1PE", "2peter": "2PE",
    "1john": "1JN", "2john": "2JN", "3john": "3JN", "jude": "JUD",
    "revelation": "REV",
}

_book_cache: dict[str, dict[int, dict[int, str]]] = {}

def _load_book(code: str) -> dict[int, dict[int, str]] | None:
    if code in _book_cache:
        return _book_cache[code]
    filename = usfm_files.get(code)
    if not filename:
        return None
    text = requests.get(RAW_BASE + filename, headers=HEADERS, timeout=15).text
    chapters: dict[int, dict[int, str]] = {}
    chapter = None
    for line in text.splitlines():
        line = line.strip()
        cm = re.match(r"\\c\s+(\d+)", line)
        if cm:
            chapter = int(cm.group(1))
            chapters.setdefault(chapter, {})
            continue
        vm = re.match(r"\\v\s+(\d+)\s*(.*)", line)
        if vm and chapter is not None:
            verse = int(vm.group(1))
            verse_text = vm.group(2)
            # Strip inline footnotes like "\f + \fr ... \ft ... \f*"
            verse_text = re.sub(r"\\f\s*\+.*?\\f\*", "", verse_text).strip()
            chapters[chapter][verse] = verse_text
    _book_cache[code] = chapters
    return chapters


def lookup_malayalam_verse(ref: str) -> str | None:
    ref = ref.replace("\xa0", " ").strip()
    m = re.match(r"^((?:[1-3]\s)?[A-Za-z.]+)\s(\d+):(\d+)(?:-(\d+))?$", ref)
    if not m:
        return None
    book_raw, chapter, verse, verse_end = m.groups()
    key = re.sub(r"[\s.]", "", book_raw).lower()
    code = BOOK_ALIASES.get(key)
    if not code:
        return None
    chapters = _load_book(code)
    if not chapters or int(chapter) not in chapters:
        return None
    verses = chapters[int(chapter)]
    lo = int(verse)
    hi = int(verse_end) if verse_end else lo
    texts = [verses[v] for v in range(lo, hi + 1) if v in verses]
    return " ".join(texts) if texts else None


Found 66 Malayalam USFM book files in the live repo.


In [6]:
for rec in records[:8]:
    print(f"headword:   {rec['headword']}")
    print(f"definition: {rec['definition'][:160]}{'...' if len(rec['definition']) > 160 else ''}")
    print("candidate Malayalam context (from cited refs):")
    if not rec["refs"]:
        print("  (no scripture refs extracted -- nothing to look up)")
    for ref in rec["refs"][:3]:
        ml_text = lookup_malayalam_verse(ref)
        if ml_text:
            print(f"  [{ref}] {ml_text}")
        else:
            print(f"  [{ref}] -- not found (book/verse mapping miss, needs a manual check)")
    print("-" * 80)


headword:   Aaron
definition: the eldest son of Amram and Jochebed, a daughter of Levi (Ex. 6:20). Some explain the name as meaning mountaineer, others mountain of strength, illuminator. He ...
candidate Malayalam context (from cited refs):
  [Ex. 6:20] -- not found (book/verse mapping miss, needs a manual check)
  [1 Chr. 2:10] -- not found (book/verse mapping miss, needs a manual check)
  [Ex. 4:14,27-30] -- not found (book/verse mapping miss, needs a manual check)
--------------------------------------------------------------------------------
headword:   Aaronites
definition: the descendants of Aaron, and therefore priests. Jehoiada, the father of Benaiah, led 3,700 Aaronites as "fighting men" to the support of David at Hebron (1 Chr...
candidate Malayalam context (from cited refs):
  [1 Chr. 12:27] -- not found (book/verse mapping miss, needs a manual check)
  [Num. 3:32] -- not found (book/verse mapping miss, needs a manual check)
  [1 Chr. 27:17] -- not found (book/verse mapping

  [Esther 1:10] ഏഴാം ദിവസം വീഞ്ഞ് കുടിച്ച് സന്തുഷ്ടനായപ്പോൾ അഹശ്വേരോശ്‌ രാജാവ്: മെഹൂമാൻ, ബിസ്ഥാ, ഹർബ്ബോനാ, ബിഗ്ദ്ധാ, അബഗ്ദ്ധാ, സേഥർ, കർക്കസ് എന്നിങ്ങനെ രാജധാനിയിൽ സേവിച്ചുനില്ക്കുന്ന
--------------------------------------------------------------------------------
headword:   Abana
definition: stony (Heb. marg. "Amanah," perennial), the chief river of Damascus (2 Kings 5:12). Its modern name is Barada, the Chrysorrhoas, or "golden stream," of the Gree...
candidate Malayalam context (from cited refs):


  [2 Kings 5:12] ദമ്മേശെക്കിലെ നദികളായ അബാനയും പർപ്പരും യിസ്രായേൽദേശത്തിലെ എല്ലാ വെള്ളത്തെക്കാളും നല്ലതല്ലയോ? എനിക്ക് അവയിൽ കുളിച്ച് ശുദ്ധനാകരുതോ?” എന്ന് പറഞ്ഞ് അവൻ ക്രോധത്തോടെ പോയി.
--------------------------------------------------------------------------------
headword:   Abarim
definition: regions beyond; i.e., on the east of Jordan, a mountain, or rather a mountain-chain, over against Jericho, to the east and south-east of the Dead Sea, in the la...
candidate Malayalam context (from cited refs):
  [Deut. 3:27] -- not found (book/verse mapping miss, needs a manual check)
  [Num. 33:47,48] -- not found (book/verse mapping miss, needs a manual check)
--------------------------------------------------------------------------------
headword:   Abba
definition: This Syriac or Chaldee word is found three times in the New Testament (Mark 14:36; Rom. 8:15; Gal. 4:6), and in each case is followed by its Greek equivalent, w...
candidate Malayalam context (from cited refs):


  [Mark 14:36] \wj “അബ്ബാ, പിതാവേ, നിനക്ക് എല്ലാം കഴിയും; ഈ പാനപാത്രം എങ്കൽ നിന്നു നീക്കേണമേ; എങ്കിലും എന്റെ ഹിതമല്ല അങ്ങയുടെ ഹിതം തന്നേ” ആകട്ടെ\wj* എന്നു പറഞ്ഞു.
  [Rom. 8:15] -- not found (book/verse mapping miss, needs a manual check)
  [Gal. 4:6] -- not found (book/verse mapping miss, needs a manual check)
--------------------------------------------------------------------------------
headword:   Abda
definition: servant. (1.) The father of Adoniram, whom Solomon set over the tribute (1 Kings 4:6); i.e., the forced labour (R.V., "levy"). (2.) A Levite of the family of Je...
candidate Malayalam context (from cited refs):


  [1 Kings 4:6] അഹീശാർ കൊട്ടാരംവിചാരകൻ; അബ്ദയുടെ മകൻ അദോനീരാം കഠിനവേല ചെയ്യുന്നവരുടെ മേധാവി.
  [Neh. 11:17] -- not found (book/verse mapping miss, needs a manual check)
  [1 Chr. 9:16] -- not found (book/verse mapping miss, needs a manual check)
--------------------------------------------------------------------------------


## What a human labeller does from here

For each entry, they read the Malayalam verse text shown above, find the
word/phrase corresponding to the English headword in context, and enter it
into `ml_variants` themselves (with the tradition/source noted if it varies
across translations). This tool only removes the "go find the right verse"
step — it makes no claim about which Malayalam word is the right one.

Known limitations to flag before relying on this at scale:
- **Versification risk** (from the original plan): if this Malayalam Bible
  edition splits chapters/verses differently from the English source
  Easton's refs were written against, a lookup can silently return the
  *wrong* verse rather than failing loudly. Worth spot-checking a handful
  of lookups against a printed/trusted Malayalam Bible before trusting this
  broadly.
- Only refs matching the simple `Book Chapter:Verse` or `Book Chapter:Verse-Verse`
  pattern resolve — multi-chapter ranges or looser citation styles
  (e.g. "Luke 3") are not handled by this regex and will just report "not
  found".
- Footnote stripping is regex-based and may leave artifacts on more complex
  verses than the ones sampled here.
